# Linear Regression — Barcelona Noise Prediction

**Goal:** Predict daytime street noise (`noise_day`, in dB) from physical and contextual
features of each street segment.

**Steps covered in this notebook:**
1. Load data
2. Select features & encode categorical columns
3. Train / test split + feature scaling
4. Train Linear Regression
5. Evaluate (MAE, R²)
6. Visualise results

## 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

## 2 — Load the Dataset

In [ ]:
df = pd.read_csv('../../data/machine_learning/bcn_noise_ml_dataset.csv')
print(f'Shape: {df.shape}')
df.head(3)

## 3 — Feature Selection & Data Encoding

### 3.1 — Select features

We use the same curated feature set identified during exploratory data analysis.
ID columns (`street_id`, `fid`, `TRAM`) are simply not included — no need to drop them.

In [ ]:
FEATURE_COLS = [
    'road_category',
    'road_width',
    'openness',
    'road_length',
    'osm_commercial_pct_50m',
    'osm_green_pct_50m',
    'osm_industrial_pct_50m',
    'osm_other_pct_50m',
    'osm_residential_pct_50m',
    'slope_pct',
    'catastral_bldg_floors_mean_50m',
    'signal_count_50',
    'poi_count_50',
    'tree_count_50',
    'transport_count_50',
    'edge_betweenness',
]

TARGET = 'noise_day'

X = df[FEATURE_COLS].copy()
y = df[TARGET]

print(f'Features: {X.shape[1]}  |  Samples: {X.shape[0]}')
print(f'Target range: {y.min()} – {y.max()} dB')

### 3.2 — Check for missing values

If any column has NaN values the model will fail. We check and fill with the column median if needed.

In [ ]:
missing = X.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else 'None — all good!')

In [ ]:
# Fill any missing values with the column median
X = X.fillna(X.median(numeric_only=True))

### 3.3 — One-Hot Encode `road_category`

`road_category` holds labels (1, 2, 3…), not quantities.
One-hot encoding turns one column into several binary columns — one per category.

In [ ]:
print('road_category values before encoding:', sorted(X['road_category'].unique()))

X = pd.get_dummies(X, columns=['road_category'], drop_first=True)

print(f'Features after encoding: {X.shape[1]}')
print('New columns:', [c for c in X.columns if 'road_category' in c])

## 4 — Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')

### 4.1 — Feature Scaling (StandardScaler)

All features are scaled to mean=0 and std=1 so no single feature dominates just because
its values happen to be large numbers.

**Important:** we fit the scaler on training data only, then apply it to test data.
This prevents any information from the test set leaking into the model.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

## 5 — Train Linear Regression

In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print('Model trained.')

## 6 — Evaluate

In [ ]:
y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f'MAE : {mae:.2f} dB   (on average, predictions are off by this many decibels)')
print(f'R²  : {r2:.3f}       (1.0 = perfect, 0.0 = no better than guessing the mean)')

## 7 — Visualise Results

### 7.1 — Predicted vs Actual

A perfect model would place all dots on the diagonal line.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_pred, alpha=0.3, s=10, color='steelblue')

lims = [y_test.min(), y_test.max()]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')

ax.set_xlabel('Actual noise_day (dB)')
ax.set_ylabel('Predicted noise_day (dB)')
ax.set_title(f'Linear Regression — Predicted vs Actual\nMAE={mae:.2f} dB  |  R²={r2:.3f}')
ax.legend()
plt.tight_layout()
plt.show()

### 7.2 — Residuals

Residuals = actual − predicted. If the model were perfect every residual would be 0.
A pattern in the residuals (e.g. a curve) means the model is missing a non-linear relationship.

In [ ]:
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Residuals vs predicted
axes[0].scatter(y_pred, residuals, alpha=0.3, s=10, color='coral')
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_xlabel('Predicted (dB)')
axes[0].set_ylabel('Residual (dB)')
axes[0].set_title('Residuals vs Predicted')

# Histogram of residuals
axes[1].hist(residuals, bins=40, color='coral', edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_xlabel('Residual (dB)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Residuals')

plt.tight_layout()
plt.show()

### 7.3 — Feature Coefficients

Each coefficient tells you how much `noise_day` changes (in dB) when that feature
increases by 1 standard deviation. Positive = louder, negative = quieter.

In [ ]:
coeff_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_
}).sort_values('coefficient', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['steelblue' if c > 0 else 'coral' for c in coeff_df['coefficient']]
ax.barh(coeff_df['feature'], coeff_df['coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (dB per std dev)')
ax.set_title('Linear Regression — Feature Coefficients')
plt.tight_layout()
plt.show()

print(coeff_df.to_string(index=False))

---

## Interpretation

| Metric | Value | What it means |
|---|---|---|
| MAE | see above | Average dB error per street |
| R² | see above | % of noise variation explained by the model |

**If R² < 0.6 or MAE > 5 dB**, the linear model is not capturing the complexity of
urban noise. Proceed to the non-linear models in `Part 4` of the proposal
(`docs/LINEAR_REGRESSION.md`) — start with Random Forest.